In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load the dataset
df = pd.read_csv('Final_News_DF_Labeled_ExamDataset.csv')

# Label Encoding for the target column 'LABEL'
label_encoder = LabelEncoder()
df['LABEL'] = label_encoder.fit_transform(df['LABEL'])

# Separate features and target
X = df.drop(columns=['LABEL'])  # Features (word count vectors)
y = df['LABEL']  # Target (encoded labels)

# Split the data into training and testing sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize the data for ANN and CNN (standard scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshaping the data for LSTM (requires 3D input)
X_train_lstm = np.reshape(X_train_scaled, (X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_lstm = np.reshape(X_test_scaled, (X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

# Print shapes of the processed data
print(f"Shape of training data (ANN/CNN): {X_train_scaled.shape}")
print(f"Shape of testing data (ANN/CNN): {X_test_scaled.shape}")
print(f"Shape of training data (LSTM): {X_train_lstm.shape}")
print(f"Shape of testing data (LSTM): {X_test_lstm.shape}")


In [ ]:
print(X_test_lstm)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Define the ANN model
ann_model = Sequential([
    Dense(128, input_dim=X_train_scaled.shape[1], activation='relu'),  # Input layer and first hidden layer
    Dense(64, activation='relu'),  # Second hidden layer
    Dense(3, activation='softmax')  # Output layer with 3 units for classification
])

# Compile the model
ann_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
ann_history = ann_model.fit(X_train_scaled, y_train, epochs=10, batch_size=32, validation_data=(X_test_scaled, y_test))

# Evaluate the model on the test set
ann_loss, ann_accuracy = ann_model.evaluate(X_test_scaled, y_test)
print(f"ANN Test Accuracy: {ann_accuracy:.4f}")

# Plot training history
import matplotlib.pyplot as plt

plt.plot(ann_history.history['accuracy'], label='Train Accuracy')
plt.plot(ann_history.history['val_accuracy'], label='Test Accuracy')
plt.title('ANN Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# For ANN, CNN, and LSTM models
y_pred = ann_model.predict(X_test_scaled)
y_pred_labels = np.argmax(y_pred, axis=1)  # Predicted class labels
conf_matrix = confusion_matrix(y_test, y_pred_labels)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title("Confusion Matrix for ANN")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten

# Define the CNN model
cnn_model = Sequential([
    Conv1D(64, 3, activation='relu', input_shape=(X_train_scaled.shape[1], 1)),  # First Conv layer
    MaxPooling1D(2),  # Max pooling
    Conv1D(128, 3, activation='relu'),  # Second Conv layer
    MaxPooling1D(2),  # Max pooling
    Flatten(),  # Flatten the output for the dense layer
    Dense(64, activation='relu'),  # Fully connected layer
    Dense(3, activation='softmax')  # Output layer
])

# Compile the model
cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
cnn_history = cnn_model.fit(X_train_scaled, y_train, epochs=10, batch_size=32, validation_data=(X_test_scaled, y_test))

# Evaluate the model on the test set
cnn_loss, cnn_accuracy = cnn_model.evaluate(X_test_scaled, y_test)
print(f"CNN Test Accuracy: {cnn_accuracy:.4f}")

# Plot training history
plt.plot(cnn_history.history['accuracy'], label='Train Accuracy')
plt.plot(cnn_history.history['val_accuracy'], label='Test Accuracy')
plt.title('CNN Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# For ANN, CNN, and LSTM models
y_pred = cnn_model.predict(X_test_scaled)
y_pred_labels = np.argmax(y_pred, axis=1)  # Predicted class labels
conf_matrix = confusion_matrix(y_test, y_pred_labels)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title("Confusion Matrix for CNN")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
from tensorflow.keras.layers import LSTM

# Define the LSTM model
lstm_model = Sequential([
    LSTM(128, activation='relu', input_shape=(X_train_lstm.shape[1], 1)),  # LSTM layer
    Dense(64, activation='relu'),  # Dense layer
    Dense(3, activation='softmax')  # Output layer
])

# Compile the model
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
lstm_history = lstm_model.fit(X_train_lstm, y_train, epochs=10, batch_size=32, validation_data=(X_test_lstm, y_test))

# Evaluate the model on the test set
lstm_loss, lstm_accuracy = lstm_model.evaluate(X_test_lstm, y_test)
print(f"LSTM Test Accuracy: {lstm_accuracy:.4f}")

# Plot training history
plt.plot(lstm_history.history['accuracy'], label='Train Accuracy')
plt.plot(lstm_history.history['val_accuracy'], label='Test Accuracy')
plt.title('LSTM Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# For ANN, CNN, and LSTM models
y_pred = lstm_model.predict(X_test_lstm)
y_pred_labels = np.argmax(y_pred, axis=1)  # Predicted class labels
conf_matrix = confusion_matrix(y_test, y_pred_labels)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title("Confusion Matrix for LSTM")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
